# [SOLUTION] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [45]:
# Only needed for Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec('pysqlite3') is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [46]:
# Import the necessary libs
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import chromadb
from tavily import TavilyClient

from lib.llm          import LLM
from lib.messages     import UserMessage, SystemMessage, AIMessage
from lib.parsers      import JsonOutputParser
from lib.tooling import tool
from lib.agents import Agent

In [47]:
# Load environment variables
load_dotenv()

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

chroma_client = chromadb.PersistentClient(path='chromadb')
collection = chroma_client.get_collection("udaplay")
print(f'Connected to internal soucre — {collection.count()} games available')

Connected to internal soucre — 15 games available


### Tools

Three tools working in a guaranteed sequence:
- **retrieve_game** — semantic search in the local ChromaDB vault
- **evaluate_retrieval** — LLM judge: are the results sufficient?
- **game_web_search** — Tavily fallback when the vault falls short

#### Retrieve Game Tool

In [48]:
@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB.
    args:
        query (str): a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    print(f'Tool retrieve_game called with query — {query}')
    results = collection.query(
        query_texts=[query],
        n_results=2,
        include=['metadatas'],
    )
    hits = []
    for meta in results['metadatas'][0]:
        hits.append({
            'Name':          meta.get('Name'),
            'Platform':      meta.get('Platform'),
            'YearOfRelease': meta.get('YearOfRelease'),
            'Publisher':     meta.get('Publisher'),
            'Description':   meta.get('Description')
        })
    return hits

# smoke-test
sample = retrieve_game('racing game on PlayStation')
print(f'retrieve_game test → {sample}')

Tool retrieve_game called with query — racing game on PlayStation
retrieve_game test → [{'Name': 'Gran Turismo 5', 'Platform': 'PlayStation 3', 'YearOfRelease': 2010, 'Publisher': 'Sony Computer Entertainment', 'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.'}, {'Name': 'Gran Turismo', 'Platform': 'PlayStation 1', 'YearOfRelease': 1997, 'Publisher': 'Sony Computer Entertainment', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.'}]


#### Evaluate Retrieval Tool

In [49]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description='True if retrieved games answer the question')
    description: int  = Field(description='Description about the evaluation result')

In [50]:
@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """
    Based on the user's question and the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
        question (str): original question from the user
        retrieved_docs (str): retrieved documents most similar to the user query

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    print(f'Tool evaluate_retrieval called with question — {question} -  and retrieved_docs {retrieved_docs}')
    llm = LLM(model='gpt-4o-mini', temperature=0.0)
    msgs = [
        SystemMessage(content=(
            'You are a retrieval quality judge.'
            'Your task is to evaluate if the documents are enough to respond the query.'
            'Give a detailed explanation, so it is possible to take an action to accept it or not.'
        )),
        UserMessage(content=f'Question: {question}\n\nRetrieved:\n{retrieved_docs}'),
    ]
    ai_msg = llm.invoke(msgs, response_format=EvaluationReport)
    return JsonOutputParser().parse(ai_msg)

#### Game Web Search Tool

In [51]:
@tool
def game_web_search(question: str) -> dict:
    """
    Search the web for information about the game industry.
    args:
        question (str): a question about the game industry.
    """
    print(f'Tool game_web_search called with question — {question}')
    client = TavilyClient(api_key=TAVILY_API_KEY)
    result = client.search(
        query             = question,
        search_depth      = 'advanced',
        include_answer    = True,
        include_raw_content = False,
    )
    return {
        "answer": result.get("answer", ""),
        "results": result.get("results", []),
    }

### Agent

```

In [52]:
udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are UdaPlay, an AI Research Agent for the video game industry. "
        "You help users find information about video games. "
        "Follow this workflow:\n"
        "1. Use retrieve_game to search the internal database.\n"
        "2. Use evaluate_retrieval to assess if the results are sufficient.\n"
        "3. If the retrieval is not useful, use game_web_search to find current info.\n"
        "4. Provide a comprehensive, accurate answer to the user.\n"
        "5. Always end your answer with a 'Source:' line citing where the information came from:\n"
        "   - If the answer is based on the internal database, write: 'Source: UdaPlay internal database.'\n"
        "   - If the answer uses web search results, write: 'Source: <URL(s) from the search results used>.'"
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)

In [53]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for query in queries:
    print(f"===================")
    print(f"Query: {query}")
    run = udaplay_agent.invoke(query=query)
    final_messages = run.get_final_state()["messages"]
    for msg in reversed(final_messages):
        if isinstance(msg, AIMessage) and msg.content:
            print(f"Answer: {msg.content}")
            break

Query: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
Tool retrieve_game called with query — Pokémon Gold and Silver release date
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
Tool evaluate_retrieval called with question — When was Pokémon Gold and Silver released? -  and retrieved_docs [{'Name': 'Pokémon Gold and Silver', 'Platform': 'Game Boy Color', 'YearOfRelease': 1999, 'Publisher': 'Nintendo', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'Name': 'Pokémon Ruby and Sapphire', 'Platform': 'Game Boy Advance', 'YearOfRelease': 2002, 'Publisher': 'Nintendo', 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.'}]
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor

### (Optional) Advanced

The cell below shows the accumulated session history.

In [54]:
# Session summary
print('\nSession History')
print('═'*68)
for i, run in enumerate(udaplay_agent.get_session_runs(), 1):
    fs = run.get_final_state()
    query = fs.get('user_query', '?')
    tokens = fs.get('total_tokens', '?')
    web_used = any(
        getattr(m, 'name', None) == 'game_web_search'
        for m in fs.get('messages', [])
    )
    print(f'  [{i}] {"WEB" if web_used else "INT":3}  tokens={tokens:>5}  {query[:52]}')


Session History
════════════════════════════════════════════════════════════════════
  [1] INT  tokens= 1969  When was Pokémon Gold and Silver released?
  [2] INT  tokens= 3199  Which one was the first 3D platformer Mario game?
  [3] WEB  tokens= 9178  Was Mortal Kombat X released for PlayStation 5?
  [4] WEB  tokens= 4890  Was Mortal Kombat X released for PlayStation 5?
